# W9-D4 概念实验：ReleaseChannel 与 TrafficPolicy 为什么需要灰度？

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：晋升一个“正式版本”是否应该立即改变生产流量？**

模拟 ReleaseChannel 的控制面指针和 TrafficPolicy 的运行时规则，验证移动 Channel 不触碰路由对象。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
@dataclass
class ReleaseChannel:
    name: str
    digest: str
    events: list = field(default_factory=list)
    def promote(self, digest, operator):
        self.events.append((self.digest, digest, operator))
        self.digest = digest

@dataclass(frozen=True)
class Route:
    revision_id: str
    revision_digest: str
    percentage: int

channel = ReleaseChannel("prod", "sha256:release-old")
policy = (Route("rev-old", "sha256:rev-old", 100),)
channel.promote("sha256:release-new", "release-manager")
print("Channel 当前指针：", channel.digest)
print("实际流量仍指向：", policy)
assert policy[0].revision_id == "rev-old" and len(channel.events) == 1

## 实验问题

**问题 2：灰度路由怎样保证同一租户不会在新旧版本之间跳变？**

用稳定哈希把 tenant 固定映射到 cohort；同一键重复请求得到同一结果。

In [ ]:
def cohort(key): return int(sha256(key.encode()).hexdigest()[:8], 16) % 100
def route(key, canary_percent=15): return "rev-new" if cohort(key) < canary_percent else "rev-old"
tenants = [f"tenant-{i:03}" for i in range(100)]
assigned = [route(t) for t in tenants]
print("新版本租户数：", assigned.count("rev-new"))
print("tenant-042 的连续请求：", [route("tenant-042") for _ in range(4)])
assert len(set(route("tenant-042") for _ in range(4))) == 1

## 实验问题

**问题 3：小流量如何限制未知缺陷的影响面？**

模拟新版本对边缘请求有更高失败率；比较全量和 10% 灰度在同一批 10,000 请求上的受影响用户数。

In [ ]:
n = 10_000
keys = [f"tenant-{i % 400}" for i in range(n)]
is_edge = rng.random(n) < .08
old_fail = rng.random(n) < .01
new_fail = (rng.random(n) < .01) | (is_edge & (rng.random(n) < .38))
full_impact = new_fail.sum()
canary_mask = np.array([route(k, 10) == "rev-new" for k in keys])
canary_impact = (new_fail & canary_mask).sum() + (old_fail & ~canary_mask).sum()
print(f"全量采用新版的失败请求：{full_impact}")
print(f"10% 灰度下的失败请求：{canary_impact}")
assert canary_impact < full_impact

## 实验问题

**问题 4：灰度数据如何支持逐步扩量决策？**

对多个 canary 百分比重复评估新版本失败率，画出风险暴露量而非“是否上线”的二元判断。

In [ ]:
rates = np.array([1, 5, 10, 25, 50, 100])
exposed = []
for pct in rates:
    mask = np.array([route(k, int(pct)) == "rev-new" for k in keys])
    exposed.append((new_fail & mask).sum())
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(rates, exposed, marker="o", color="#d95f02")
ax.set_xlabel("新 Revision 流量比例（%）"); ax.set_ylabel("观测到的新版失败请求数")
ax.set_title("灰度把风险暴露量变为可逐级控制的变量")
ax.grid(alpha=.25); plt.tight_layout(); plt.show()
print(dict(zip(rates, exposed)))